<a href="https://colab.research.google.com/github/sabharwal-monish/LLM/blob/main/LLM_Training_in_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Dataset from Deep Lake

In [1]:
!pip install transformers torch

In [2]:
!pip install "deeplake<4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.4/643.4 kB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 5.4 MB/s eta 0:00:

# Load Dataset from Deep Lake

In [3]:
import deeplake

ds = deeplake.load('hub://activeloop/openwebtext-train', read_only=True)
ds_val = deeplake.load('hub://activeloop/openwebtext-val', read_only=True)

print("Connection Successful!")
print(ds)
print(ds[0].text.text())

/usr/local/lib/python3.12/dist-packages/deeplake/util/check_latest_version.py:32: UserWarning: A newer version of deeplake (4.4.2) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(
\

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/openwebtext-train



-

hub://activeloop/openwebtext-train loaded successfully.



|

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/openwebtext-val



-

hub://activeloop/openwebtext-val loaded successfully.



Connection Successful!
Dataset(path='hub://activeloop/openwebtext-train', read_only=True, tensors=['text', 'tokens'])
An in-browser module loader configured to get external dependencies directly from CDN. Includes babel/typescript. For quick prototyping, code sharing, teaching/learning - a super simple web dev environment without node/webpack/etc.

All front-end libraries

Angular, React, Vue, Bootstrap, Handlebars, jQuery are included. Plus all packages from cdnjs.com and all of NPM (via unpkg.com). Most front-end libraries should work out of the box - just use import / require() . If a popular library does not load, tell us and we’ll try to solve it with some library-specific config.

Write modern javascript (or typescript)

Use latest language features or JSX and the code will be transpiled in-browser via babel or typescript (if required). To make it fast the transpiler will start in a worker thread and only process the modified code. Unless you change many files at once or open the

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [14]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

  def __init__(self, ds):
    self.ds = ds

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, idx):
    tokenized_text = tokenizer(
        self.ds.text[idx].text(),
        truncation = True,
        max_length = 512,
        padding = 'max_length',
        return_tensors = 'pt'
    )

    tokenized_text = tokenized_text['input_ids'][0]

    sample = {'input_ids': tokenized_text, 'labels': tokenized_text }
    return sample



In [15]:
myTrainingLoader = MyDataset(ds)
myValidationLoader = MyDataset(ds_val)

# Load the Model

In [16]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained('gpt2')
print(config)

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.1",
  "use_cache": true,
  "vocab_size": 50257
}



In [17]:
from transformers import GPT2LMHeadModel
from accelerate import Accelerator

model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f'GPT2 Size:{model_size/1e6:.1f} M Parameters')


GPT2 Size:124.4 M Parameters


# Training

In [22]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="GPT2-scratch-openwebtext",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,


    max_steps=500,
    num_train_epochs=2,


    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,


    bf16=False,
    fp16=True,

    ddp_find_unused_parameters=False,
    run_name="GPT2-scratch-openwebtext",
    report_to="none"
)

print("Arguments loaded successfully!")

Arguments loaded successfully!


In [23]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=myTrainingLoader,
    eval_dataset=myValidationLoader,
)

In [24]:
from transformers import Trainer
trainer.train()

Step,Training Loss,Validation Loss
100,6.216200,6.645453
200,6.250200,6.550663
300,5.719500,6.243855
400,6.078600,6.164234
500,5.984300,6.139860


/usr/local/lib/python3.12/dist-packages/deeplake/core/tensor.py:719: UserWarning: Indexing by integer in a for loop, like `for i in range(len(ds)): ... ds.tensor[i]` can be quite slow. Use `for i, sample in enumerate(ds)` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/deeplake/core/tensor.py:719: UserWarning: Indexing by integer in a for loop, like `for i in range(len(ds)): ... ds.tensor[i]` can be quite slow. Use `for i, sample in enumerate(ds)` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/deeplake/core/tensor.py:719: UserWarning: Indexing by integer in a for loop, like `for i in range(len(ds)): ... ds.tensor[i]` can be quite slow. Use `for i, sample in enumerate(ds)` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/deeplake/core/tensor.py:719: UserWarning: Indexing by integer in a for loop, like `for i in range(len(ds)): ... ds.tensor[i]` can be quite slow. Use `for i, sample in enumerate(ds)` instead.
  warnings.warn(
/usr/loc

TrainOutput(global_step=500, training_loss=6.158767318725586, metrics={'train_runtime': 1230.9794, 'train_samples_per_second': 0.406, 'train_steps_per_second': 0.406, 'total_flos': 130646016000000.0, 'train_loss': 6.158767318725586, 'epoch': 6.242382732470703e-05})

# Inference

In [31]:

save_path = "./my_final_gpt2"

print("Saving model to disk...")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Save complete!")




from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

print("Loading pipeline...")
pipe = pipeline("text-generation",
                model=save_path,
                tokenizer=tokenizer,
                device=device)


print("Generating text...")
txt = "The house prices dropped down"
completion = pipe(txt, num_return_sequences=1)
print(completion)

Saving model to disk...


Device set to use cuda:0


Save complete!
Loading pipeline...
Generating text...
[{'generated_text': "The house prices dropped down the two of the new-I was more from the way to be the former year of an two-year, which will make a world to the state's a day of the third months of the time.\n\n\n\n\n\n\n\n\n\n\nWe don had a most of the new-year, the last week, at the former-I think they't the end.\n\nThe top of the past to the American of a company, the own-He’m.\n\n\nIf for the last team. The month.\n\n\n\n\n\nThe new time.\nThe new of the full,” and the big.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n’re in many that this of the former-I’s one,’s the year for the US, the bit of a day to a world of the most as that the own-If just see. The first, and the first, the world of the same, and the most right to being do a second.\n\n“I’s the own of the two are the own of the year.’s it’s time. It’"}]
